# Project 1: Cybersecurity Alert Triage (SOC Operations)

**Synthetic Dataset Specification & Mathematical Blueprint — v2 (corrected)**

---

| Item | Detail |
|:---|:---|
| **Author** | *Mark Christian Anub* |
| **Date** | *June 2026* |
| **Status** | Draft v2 (math reconciled, alert + BTP mechanisms added) |
| **Target Learner** | SOC Analyst / Cybersecurity Data Scientist |
| **Tables** | 6 core tables, ~44 features |
| **Latent (omitted) variables** | `hidden_compromise_state` (carries episode *type*) + `hidden_analyst_fatigue` |

---

## 1. Business Context & Challenge Statement

Security Operations Centers (SOCs) suffer from severe **"Alert Fatigue."** Enterprise SIEMs (Security Information and Event Management systems) generate tens of thousands of alerts daily, of which **95–98% are false positives** caused by misconfigured rules or benign anomalies. Meanwhile, actual threat actors execute multi-stage attack chains (e.g., credential theft $\rightarrow$ lateral movement $\rightarrow$ data exfiltration) that hide within this noise.

### 1.1 The Learner's Challenge
You are a Data Scientist hired by a Fortune 500 company's SOC. You have been provided with **30 days** of messy network telemetry, SIEM alerts, and analyst response logs across **6 relational tables**. Your goal is to build a Machine Learning classification model to predict the `final_verdict` of an alert:
- **True Positive (TP):** A genuine cybersecurity breach requiring immediate incident response.
- **False Positive (FP):** A benign alert caused by misconfigured rules, routine scans, or non-malicious anomalies.
- **Benign True Positive (BTP):** A real anomaly that is *authorized* (e.g., a sanctioned penetration test). It looks identical to an attack in raw telemetry but represents no actual threat.

### 1.2 What Makes This Dataset Unique

> ** Warning to Learners:** This dataset contains realistic *human errors*. SOC analysts working long shifts occasionally mislabel true breaches as false positives due to cognitive fatigue. Furthermore, the strongest numeric signals (like transfer bytes and log volume) deliberately overlap between safe and compromised states. Models that trust the labels blindly or rely on single-feature thresholds **will fail**.

## 2. Data Schema

The synthetic environment consists of **6 relational tables** with **~44 total features** and **2 hidden latent variables** (dropped before export).

### 2.1 Schema Overview

| # | Table | PK | FK(s) | Scale | Purpose |
|:---|:---|:---|:---|:---|:---|
| 1 | `users` | `user_id` | — | ~200–500 | Employee directory & behavioral baselines |
| 2 | `devices` | `device_id` | `user_id` | ~300–800 | Network assets & criticality ratings |
| 3 | `log_events` | `event_id` | `device_id`, `user_id` | ~500K–5M | High-volume time-series telemetry |
| 4 | `alerts` | `alert_id` | `device_id` | ~10K–50K | SIEM-aggregated security alerts |
| 5 | `response_actions` | `action_id` | `alert_id` | ~10K–50K | SOC analyst triage actions & shift metadata |
| 6 | `alert_outcomes` | `alert_id` | `alert_id` | ~10K–50K | Ground-truth labels & financial impact |

### 2.2 Entity Relationships
- `users` **(1)** $\rightarrow$ **(N)** `devices` — Each employee is assigned one or more devices.
- `users` **(1)** $\rightarrow$ **(N)** `log_events` — Events are traced back to the acting user.
- `devices` **(1)** $\rightarrow$ **(N)** `log_events` — Events originate from a specific device.
- `devices` **(1)** $\rightarrow$ **(N)** `alerts` — Alerts are triggered on a specific device.
- `alerts` **(1)** $\rightarrow$ **(1)** `response_actions` — Each alert receives exactly one triage response.
- `alerts` **(1)** $\rightarrow$ **(1)** `alert_outcomes` — Each alert has exactly one final verdict.

---

### 2.3 Table: `users` (Employee Directory & Behavioral Baselines)

| # | Feature | Data Type | Description | Example Values / Range |
|:---|:---|:---|:---|:---|
| 1 | `user_id` | `string` (PK) | Unique employee identifier | `USR-0001`, `USR-0002` |
| 2 | `username` | `string` | Employee display name | `alice.chen`, `bob.martinez` |
| 3 | `role` | `categorical` | Access privilege level | `Standard_User`, `Admin`, `Privileged_User` |
| 4 | `department` | `categorical` | Organizational unit | `IT`, `Finance`, `HR`, `Engineering`, `Executive` |
| 5 | `hire_date` | `date` | Employment start date | `2019-03-15` |
| 6 | `typical_login_start` | `int` (hour) | Expected shift start hour | `7`, `8`, `9`, `22` (night shift) |
| 7 | `typical_login_end` | `int` (hour) | Expected shift end hour | `16`, `17`, `18`, `6` (night shift) |
| 8 | `vpn_flag` | `boolean` | Regular remote/VPN user | `True`, `False` |

---

### 2.4 Table: `devices` (Network Topology & Asset Criticality)

| # | Feature | Data Type | Description | Example Values / Range |
|:---|:---|:---|:---|:---|
| 1 | `device_id` | `string` (PK) | Unique device identifier | `DEV-0001`, `DEV-0002` |
| 2 | `user_id` | `string` (FK) | Primary assigned user | `USR-0001` |
| 3 | `device_type` | `categorical` | Hardware category | `Laptop`, `Desktop`, `Server`, `Database_Server`, `IoT_Sensor` |
| 4 | `os_version` | `categorical` | Operating system | `Windows_11`, `Ubuntu_22.04`, `macOS_14`, `CentOS_7` |
| 5 | `criticality_score` | `int` | Asset importance (1=low, 10=critical) | `1`–`10` |
| 6 | `department` | `categorical` | Department (inherited from user) | `IT`, `Finance`, `HR`, `Engineering` |
| 7 | `ip_address` | `string` | Internal network IP | `10.0.1.45`, `192.168.5.12` |

### 2.5 Table: `log_events` (High-Volume Network Telemetry)

*This is the largest table — the raw, noisy, time-series data that the learner must mine for attack signals.*

| # | Feature | Data Type | Description | Example Values / Range |
|:---|:---|:---|:---|:---|
| 1 | `event_id` | `string` (PK) | Unique event identifier | `EVT-0000001` |
| 2 | `timestamp` | `datetime` | Event time (with ±3s jitter) | `2026-06-01 14:23:07.412` |
| 3 | `device_id` | `string` (FK) | Source device | `DEV-0042` |
| 4 | `user_id` | `string` (FK) | Acting user | `USR-0015` |
| 5 | `event_type` | `categorical` | Type of network event | `Login_Success`, `Login_Fail`, `File_Download`, `File_Upload`, `Outbound_Connection`, `Process_Start`, `RDP_Session`, `Firewall_Block` |
| 6 | `source_ip` | `string` | Origin IP address (~3–5% missing) | `10.0.1.45`, `NaN` |
| 7 | `dest_ip` | `string` | Destination IP address | `10.0.2.100`, `203.0.113.50` |
| 8 | `bytes_transferred` | `float` | Data volume in bytes (drawn from state mixture) | See mathematical definition in §4.3 |
| 9 | `geo_location` | `categorical` | Country of source IP | `PH`, `SG`, `US`, `DE`, `RU`, `CN` |
| 10 | `process_name` | `categorical` | Executable that triggered the event | See process list below |
| 11 | `session_duration_sec` | `float` | Duration of session/connection | `0.5`–`28800` (8 hours) |

#### Process Name Reference List

| Category | Process Names |
|:---|:---|
| **Normal IT** | `chrome.exe`, `outlook.exe`, `teams.exe`, `excel.exe`, `svchost.exe`, `explorer.exe`, `python.exe`, `code.exe`, `sqlservr.exe`, `nginx`, `sshd`, `systemd` |
| **Potentially Suspicious** | `powershell.exe`, `cmd.exe`, `wscript.exe`, `certutil.exe`, `curl.exe` |
| **Malicious (Red Team)** | `mimikatz.exe`, `psexec.exe`, `sharphound.exe`, `lazagne.exe`, `procdump.exe` |

> **Realism Note:** Malicious process names appear *almost exclusively* when `hidden_compromise_state = 1`. However, `powershell.exe` and `cmd.exe` also appear in ~15% of normal operations to prevent trivial detection.

---

### 2.6 Table: `alerts` (SIEM Aggregated Security Alerts)

| # | Feature | Data Type | Description | Example Values / Range |
|:---|:---|:---|:---|:---|
| 1 | `alert_id` | `string` (PK) | Unique alert identifier | `ALR-00001` |
| 2 | `timestamp` | `datetime` | Alert generation time | `2026-06-01 14:25:00` |
| 3 | `device_id` | `string` (FK) | Device that triggered the alert | `DEV-0042` |
| 4 | `alert_rule_triggered` | `categorical` | Detection rule name | `Brute_Force`, `Data_Exfil_Suspected`, `Malware_Signature`, `Impossible_Travel`, `Unusual_Process`, `Port_Scan_Detected` |
| 5 | `severity` | `categorical` | Alert severity level (conditioned on source) | `Low`, `Medium`, `High`, `Critical` |
| 6 | `mitre_tactic` | `categorical` | MITRE ATT&CK tactic | `Reconnaissance`, `Initial_Access`, `Execution`, `Lateral_Movement`, `Exfiltration`, `Command_and_Control` |
| 7 | `confidence_score` | `float` | SIEM confidence (Beta distribution) | See conditional distributions in §4.5 |
| 8 | `alert_source` | `categorical` | Detection tool | `Firewall`, `EDR`, `IDS`, `Email_Gateway` |

---

### 2.7 Table: `response_actions` (SOC Analyst Triage)

| # | Feature | Data Type | Description | Example Values / Range |
|:---|:---|:---|:---|:---|
| 1 | `action_id` | `string` (PK) | Unique action identifier | `ACT-00001` |
| 2 | `alert_id` | `string` (FK) | Associated alert | `ALR-00001` |
| 3 | `analyst_id` | `string` | SOC analyst who handled it | `SOC_T1_Alice`, `SOC_T2_Bob` |
| 4 | `action_taken` | `categorical` | Triage decision | `Isolate_Host`, `Block_IP`, `Reset_Credentials`, `Close_False_Positive`, `Escalate_to_Tier2` |
| 5 | `time_to_respond_mins` | `float` | Minutes to first action (Log-Normal, affected by fatigue) | `2.5`–`480.0` |
| 6 | `shift_time` | `categorical` | Analyst's current shift | `Day_Shift` (06:00–18:00), `Night_Shift` (18:00–06:00) |
| 7 | `alerts_handled_today` | `int` | Running count of alerts handled this shift | `1`–`100+` |

---

### 2.8 Table: `alert_outcomes` (Ground-Truth Labels & Business Impact)

| # | Feature | Data Type | Description | Example Values / Range |
|:---|:---|:---|:---|:---|
| 1 | `alert_id` | `string` (PK/FK) | Associated alert | `ALR-00001` |
| 2 | `final_verdict` | `categorical` | Ground-truth classification (with label noise) | `True_Positive` (~3%), `False_Positive` (~95%), `Benign_True_Positive` (~2%) |
| 3 | `business_impact_usd` | `float` | Estimated financial impact (based on true state) | See mathematical definition in §4.6 |

## 3. Realism & Messiness Injection Strategy

Real-world cybersecurity data is messy. The following artifacts are **intentionally injected** into the generated dataset to simulate production conditions:

### 3.1 Missing Values

| Table | Field | Missing Rate | Rationale |
|:---|:---|:---|:---|
| `log_events` | `source_ip` | ~3–5% | Firewalls under heavy load drop source IP metadata |
| `log_events` | `bytes_transferred` | ~1–2% | Some event types (e.g., `Login_Fail`) don't log payload size |
| `response_actions` | `time_to_respond_mins` | ~2% | Automated responses (scripts) don't record human response time |

### 3.2 Timestamp Jitter
- All `log_events.timestamp` values have **±3 seconds** of uniform random noise added.
- **Rationale:** Enterprise networks consist of hundreds of devices with imperfectly synchronized NTP clocks. Timestamps are never perfectly aligned in real SIEM data.

### 3.3 Class Imbalance
- `alert_outcomes.final_verdict` distribution:
  - **~95% False Positive** — The overwhelming majority of alerts are noise.
  - **~3% True Positive** — Actual breaches are rare.
  - **~2% Benign True Positive** — Real but harmless events (e.g., authorized pen tests).
- **Rationale:** Industry reports confirm that enterprise SOCs face 95–99% false positive rates.

### 3.4 Duplicate Alerts
- ~5% of alerts are **near-duplicates** (same device, same rule, within ±60 seconds).
- **Rationale:** SIEMs often fire the same detection rule multiple times for a single underlying event.

### 3.5 Seasonal & Temporal Patterns
- **Monday mornings** see a spike in phishing-related alerts (employees returning from weekends).
- **Friday afternoons** see reduced analyst response quality (pre-weekend fatigue).
- **3:00 AM–5:00 AM** windows have elevated attack activity (adversaries prefer off-hours).
- **Rationale:** Both attacker behavior and defender fatigue follow predictable temporal patterns documented in threat intelligence reports.

### 3.6 Impossible Travel Injection
- During active compromise periods ($C_t = 1$), some `log_events` show the same `user_id` logging in from geographically impossible locations within short time windows (e.g., `PH` and `RU` within 5 minutes).
- **Rationale:** Impossible travel is a classic indicator of compromised credentials used simultaneously by the real user and the attacker.

### 3.7 Feature Overlap (Bytes Transferred Mixture)
- Instead of using a simple cut-off threshold for file sizes, the `bytes_transferred` field is drawn from overlapping, multi-component distributions. Large legitimate transfers (e.g., system backups, large downloads) collide with stealthy "low-and-slow" exfiltration, rendering trivial single-variable classification rules useless.

## 4. Mathematical Model & Structural Equations

### 4.1 Hidden Anomaly State ($C_{d,t}$) — The Puppet Master (typed)

The anomaly state is a binary latent variable that governs the entire simulation. For each device $d$ in the network during hour $t$:

$$C_{d,t} \in \{0, 1\}$$

- $C_{d,t} = 0$: Device is operating normally (safe state).
- $C_{d,t} = 1$: An anomalous episode is active on the device.

When an anomalous episode is seeded, it is tagged with a latent category:

$$\texttt{episode\_type} = \begin{cases} \text{Authorized} & \text{with probability } p_{auth} = 0.35 \\ \text{Malicious} & \text{with probability } 1 - p_{auth} = 0.65 \end{cases}$$

**Both episode types generate identical network telemetry** (volume spikes, red-team tools, exfiltration bytes). They differ only in ground truth: `Malicious` translates to a `True_Positive` final outcome, whereas `Authorized` translates to a `Benign_True_Positive` (e.g., a sanctioned penetration test).

Once a device enters an active episode, the state persists for a **dwell time** $T_{dwell}$:

$$T_{dwell} \sim \text{LogNormal}(\mu = 4.5, \, \sigma = 1.0) \quad \text{(in hours)}$$

This yields a median dwell time of $e^{4.5} \approx 90$ hours (~3.75 days). Per the 2023 Mandiant M-Trends Report, the global median dwell time is ~16 days. We compress this to hours for a 30-day simulation window.

---

### 4.2 Network Telemetry Volume — Poisson Process

The arrival rate of log events $\lambda_{d,t}$ for device $d$ at hour $t$ is modeled as a **conditional Poisson process**:

$$\lambda_{d,t} = \lambda_{base} \cdot (1 + \gamma \cdot C_{d,t})$$

| Parameter | Value | Meaning |
|:---|:---|:---|
| $\lambda_{base}$ | 10 events/hour | Normal baseline traffic rate |
| $\gamma$ | 9 | Lateral movement multiplier (10× spike during compromises) |

**Interpretation:**
- **Safe state** ($C_{d,t} = 0$): $\lambda_{d,t} = 10 \cdot (1 + 9 \cdot 0) = 10$ events/hour.
- **Compromised** ($C_{d,t} = 1$): $\lambda_{d,t} = 10 \cdot (1 + 9 \cdot 1) = 100$ events/hour.

**Row-Count Verification:**
With $N_{devices} \approx 400$ and $T_{hours} = 720$ (30 days), the expected baseline row count is:
$$\mathbb{E}[\text{Rows}] \approx 400 \times 720 \times 10 = 2,880,000 \text{ rows}$$
This comfortably aligns with the declared limits of $500\text{K} - 5\text{M}$ rows.

---

### 4.3 Data Exfiltration Payloads — Log-Normal Mixture

For each event, the bytes transferred $B$ is drawn from a state-conditioned **two-component mixture model**. Recall that for $\text{LogNormal}(\mu, \sigma)$, the median is $e^{\mu}$ and the mean is $e^{\mu + \sigma^2/2}$.

**Safe State ($C_{d,t} = 0$):**

$$B \sim \begin{cases} \text{LogNormal}(\mu=11,\ \sigma=1.0) & \text{with probability } 0.90 \quad (\text{median} \approx 60 \text{ KB; normal browsing}) \\ \text{LogNormal}(\mu=16,\ \sigma=1.0) & \text{with probability } 0.10 \quad (\text{median} \approx 8.9 \text{ MB; backups/video}) \end{cases}$$

**Compromised State ($C_{d,t} = 1$):**

$$B \sim \begin{cases} \text{LogNormal}(\mu=19,\ \sigma=1.5) & \text{with probability } 0.55 \quad (\text{median} \approx 178 \text{ MB; bulk database exfil}) \\ \text{LogNormal}(\mu=13,\ \sigma=1.5) & \text{with probability } 0.45 \quad (\text{median} \approx 442 \text{ KB; "low-and-slow" staging}) \end{cases}$$

This mixture forces the machine learning model to combine raw size with context (timing, source, device criticality), as the 10% normal backup mode (8.9 MB median) overlaps with the 45% stealthy exfiltration mode (442 KB median).

---

### 4.4 Human-in-the-Loop: Analyst Fatigue & Two-Directional Label Noise

Let $H_i$ be the cumulative number of alerts handled by analyst $i$ during their current shift. The **latent fatigue score** $F_i \in [0, 1]$ is:

$$F_i = \min\left(1.0, \, \frac{H_i}{K}\right) \quad \text{where } K = 50$$

**Impact on Response Time:**

$$\ln(\text{time\_to\_respond\_mins}) \sim \mathcal{N}(\mu_{base} + \beta_1 \cdot F_i, \, \sigma^2_{resp})$$

| Parameter | Value | Meaning |
|:---|:---|:---|
| $\mu_{base}$ | 2.0 | Base log-response time (~7.4 minutes) |
| $\beta_1$ | 1.5 | Fatigue penalty coefficient |
| $\sigma_{resp}$ | 0.8 | Response time variance |

**Label Corruption (Two-Directional Noise):**

*1. Missed Breach (Primary Effect: TP $\rightarrow$ FP)*
If an alert is a true anomaly ($C_t = 1$), the probability of a fatigued analyst mislabeling it as a False Positive is:
$$P(\text{Verdict} = \text{FP} \mid \text{True State} = \text{TP}) = \alpha_1 + \beta_2 \cdot F_i, \, \alpha_1 = 0.05, \, \beta_2 = 0.75$$
At $F_i = 0$, the error rate is 5%. At $F_i = 1$, the error rate reaches 80%.

*2. Over-Escalation (Secondary Effect: FP $\rightarrow$ TP)*
If an alert is benign ($C_t = 0$), the probability of a fatigued analyst panic-mislabeled it as a True Positive is:
$$P(\text{Verdict} = \text{TP} \mid \text{True State} = \text{FP}) = \alpha_2 + \beta_3 \cdot F_i, \, \alpha_2 = 0.01, \, \beta_3 = 0.05$$

---

### 4.5 Alert Generation

Alerts are not 1:1 with log events. For device $d$ at hour $t$, security alerts arrive as a Poisson process driven by two parameters:

$$\lambda^{alert}_{d,t} = \lambda_{FP} + \delta \cdot C_{d,t}$$

| Parameter | Value | Meaning |
|:---|:---|:---|
| $\lambda_{FP}$ | 0.05 / hour | Base rate of false alarms (always-on noise) |
| $\delta$ | 0.40 / hour | Active signal rate during compromises |

*   **Noise alerts** ($\lambda_{FP}$): Fire randomly across all devices regardless of state. Ground truth = `False_Positive`.
*   **Signal alerts** ($\delta \cdot C_{d,t}$): Fire only during active anomaly windows. Ground truth = `True_Positive` or `Benign_True_Positive` based on `episode_type`.

**Alert Attributes (Overlapping distributions):**

*   `confidence_score` is drawn from a bounded **Beta distribution**:
    $$\text{Noise alerts: } \text{Beta}(\alpha=2, \beta=6) \quad (\text{mean} \approx 0.25)$$
    $$\text{Signal alerts: } \text{Beta}(\alpha=6, \beta=2) \quad (\text{mean} \approx 0.75)$$

*   `severity` probability assignment table:
    | Source | Low | Medium | High | Critical |
    |:---|:---|:---|:---|:---|
    | **Noise** | 0.55 | 0.30 | 0.13 | 0.02 |
    | **Signal** | 0.05 | 0.20 | 0.45 | 0.30 |

---

### 4.6 Business Impact

To ensure realistic financial reporting, the estimated financial impact is conditioned on the **true underlying state** of the compromise, rather than the analyst's potentially corrupted label. If an analyst misses an ongoing breach, the financial impact remains high.

$$\texttt{business\_impact\_usd} = \begin{cases} 0 & \text{if True State is Safe } (C_{d,t} = 0) \text{ or Authorized} \\ \text{LogNormal}(\mu_I = 14, \, \sigma_I = 1.5) \cdot \dfrac{\text{criticality\_score}}{5} & \text{if True State is Malicious} \end{cases}$$

For an asset of average criticality (score = 5), this yields a median impact of $e^{14} \approx \$1.2\text{M}$ and a mean impact of $e^{14 + 1.125} \approx \$3.7\text{M}$.

## 5. Sampling / Generation Order

The generator must follow this **strict dependency order** to ensure referential integrity and statistical consistency:

| Step | Action | Depends On | Output |
|:---|:---|:---|:---|
| **1** | Generate `users` table | Config only (n_users, departments, roles) | `users.csv` |
| **2** | Generate `devices` table | `users` (FK: user_id) | `devices.csv` |
| **3** | Initialize `hidden_compromise_state` $C_{d,t}$ **and tag `episode_type`** | `devices`, Config (n_compromises, dwell_time params) | Latent arrays |
| **4** | Generate `log_events` table (mixture model §4.3, Poisson volume §4.2) | `devices`, `users`, $C_{d,t}$ | `log_events.csv` |
| **5** | Generate `alerts` table via §4.5 (noise + signal streams) | `log_events`, rules, $C_{d,t}$ | `alerts.csv` |
| **6** | Generate `response_actions` and compute `hidden_analyst_fatigue` $F_i$ | `alerts`, analyst pool, shift schedule | `response_actions.csv` + latent $F_i$ |
| **7** | Generate `alert_outcomes` (verdict corrupted by fatigue §4.4, impact derived from true state §4.6) | `alerts`, $C_{d,t}$, `episode_type`, $F_i$ | `alert_outcomes.csv` |
| **8** | **Drop** all hidden/latent columns and export | — | Final clean export |

## 6. Literature & Desk Research

The parameters and distributions chosen for this generator are grounded in industry cybersecurity reports and academic research:

### 6.1 Class Imbalance & False Positive Rates
According to a study by the Ponemon Institute (2020), enterprise SOCs suffer from intense alert volume, with security teams often unable to triage up to 45% of daily alerts due to resource constraints. Additionally, specialized detection systems report false positive rates between 95–99% of all raw triggers. This justifies our baseline setting of a ~3% True Positive rate.

> Ponemon Institute. (2020). *The economics of security operations centers: What is the true cost for effective results?* Ponemon Institute LLC.

### 6.2 Attack Chains & Lateral Movement (MITRE ATT&CK)
The sequence of `mitre_tactic` variables and the 10× traffic multiplier ($\gamma = 9$) for compromised devices are based on the MITRE ATT&CK framework's documentation of enterprise breach lifecycles. Compromised credentials typically lead to rapid internal reconnaissance, lateral movement across network segments, and eventual data exfiltration.

> MITRE Corporation. (2023). *MITRE ATT&CK® framework: Enterprise matrix* (v13). https://attack.mitre.org/

### 6.3 Data Exfiltration Volumes & Breach Costs
The Verizon DBIR notes that the majority of breaches involve the exfiltration of large databases or credential stores, manifesting as heavy-tailed distributions in network egress traffic. This justifies our Log-Normal mixture model for compromised states. The IBM Cost of a Data Breach Report (2022) reports an average breach cost of \$4.35M USD, informing our `business_impact_usd` distribution.

> Verizon Enterprise. (2023). *2023 data breach investigations report (DBIR)*. Verizon Communications.
>
> IBM Security. (2022). *Cost of a data breach report 2022*. IBM Corporation.

### 6.4 Analyst Fatigue & Human Error in SOCs
Research on SOC analyst burnout shows that cognitive load directly correlates with misclassification errors during triage. Studies indicate that after handling 40–60 alerts in a single shift, analyst accuracy degrades significantly. Our fatigue threshold ($K = 50$) and penalty coefficient ($\beta_2 = 0.75$) simulate this cognitive degradation.

> Sundaramurthy, S. C., McHugh, J., Ou, X., Wesch, M., Bardas, A. G., & Rajagopalan, S. R. (2016). Turning contradictions into innovations or: How we learned to stop whining and improve security operations. *Proceedings of the 12th Symposium on Usable Privacy and Security (SOUPS)*, 237–251.

### 6.5 Dwell Time
The 2023 Mandiant M-Trends report indicates a global median dwell time of 16 days for externally detected intrusions. Our compressed Log-Normal dwell time ($\mu = 4.5$ hours) is scaled proportionally for the 30-day simulation window.

> Mandiant. (2023). *M-Trends 2023: Special report*. Google Cloud / Mandiant.

## 7. Omitted Variables Declaration

To ensure the dataset requires genuine feature engineering and prevents trivial solutions, the following **latent variables** are generated in memory but are **strictly dropped** before the final CSV export:

---

### 7.1 `hidden_compromise_state` ($C_{d,t}$) + `episode_type`

| Property | Detail |
|:---|:---|
| **Type** | Binary (0 = Safe, 1 = Compromised) paired with categorical (`Malicious`/`Authorized`) |
| **Scope** | Per-device, per-hour time series |
| **Drives** | Log volume (Poisson $\lambda$), `bytes_transferred` mixture, malicious `process_name` probability, `alert` generation rate, `final_verdict` ground truth |

**Why it's omitted:** In the real world, a SOC analyst does not have a column that says "this device is hacked." The learner must infer compromise from secondary signals:
- Sudden spikes in log volume on a single device
- Anomalous `bytes_transferred` values (heavy tail)
- Appearance of red-team tools (`mimikatz.exe`, `psexec.exe`)
- Impossible travel patterns in `geo_location`
- Temporal clustering of high-severity alerts

*TP vs BTP is intentionally not separable from telemetry alone.* This ambiguity mimics the real-world operational challenges of security analysts.

---

### 7.2 `hidden_analyst_fatigue` ($F_i$)

| Property | Detail |
|:---|:---|
| **Type** | Continuous float [0.0, 1.0] |
| **Scope** | Per-analyst, per-shift |
| **Drives** | `time_to_respond_mins` inflation, `final_verdict` mislabeling probability |

**Why it's omitted:** Real-world datasets suffer from **label noise** due to human error, but the source of that noise is never explicitly recorded. By hiding the fatigue score, the learner's ML model will experience unexplained variance:
- "Why did my model miss this obvious breach?"
- "Why are there clusters of mislabeled alerts at certain times of day?"

**Detection strategies for advanced learners:**
- Rolling count of `alerts_handled_today` per `analyst_id`
- Interaction between `shift_time` (Night vs. Day) and `final_verdict` accuracy
- Time-of-day analysis on `time_to_respond_mins` distributions

---

### Summary of Omitted Variable Effects

| Omitted Variable | Observable Effect in Final Data | Detection Strategy |
|:---|:---|:---|
| `hidden_compromise_state` | Correlated spikes in log volume, bytes, malicious processes, and alerts on the same device within a time window | Device-level time-series aggregation + anomaly detection |
| `hidden_analyst_fatigue` | Clusters of mislabeled True Positives during night shifts or after high alert counts | Analyst-level rolling statistics + shift-based stratification |